In [0]:
%run /Users/rahulpatel@cyntexa.com/IAP_Practice/freshkart/00_set_up

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# --- F3: Liquid clustering on gold.fact_order_line ---
# Apply liquid clustering on (order_date_key, store_id), then optimize existing data.

# 1. Apply liquid clustering keys
spark.sql(f"ALTER TABLE {catalog}.gold.fact_order_line CLUSTER BY (order_date_key, store_id)")

# 2. Optimize — re-cluster existing data (ensures at least one optimization pass)
spark.sql(f"OPTIMIZE {catalog}.gold.fact_order_line")

# # 3. Verify clustering configuration
# print("=== Clustering configuration ===")
# display(spark.sql(f"DESCRIBE DETAIL {catalog}.gold.fact_order_line").select("clusteringColumns", "clusterByAuto"))

In [0]:
# --- Recovery Drill: Deep clone → Delete → Restore via Delta time travel ---

# Step 0: Drop any previous drill table
spark.sql(f"DROP TABLE IF EXISTS {catalog}.ops.orders_drill")

# Step 1: Create a deep clone of silver.orders
spark.sql(f"CREATE TABLE {catalog}.ops.orders_drill DEEP CLONE {catalog}.silver.orders")


spark.sql(f"DELETE FROM {catalog}.ops.orders_drill WHERE store_id IN ('S01', 'S02')")


# spark.sql(f"RESTORE TABLE {catalog}.ops.orders_drill TO VERSION AS OF {1}")



In [0]:
%sql 

DESCRIBE HISTORY fk_t12.ops.orders_drill;

RESTORE TABLE fk_t12.ops.orders_drill TO VERSION AS OF 0;